In [1]:
!pip install pyarrow deltalake jupyterlab-rise -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


### Import libraries and configure input paths  


In [2]:
import json
import time
from pathlib import Path
from collections import Counter
import pandas as pd
import duckdb
import os

In [ ]:
#Input Paths
FHIR_DIR = Path("synthea/output/fhir")
OUT_DIR = Path("puppygraph_output")
OUT_DIR.mkdir(exist_ok=True)

### Structure of a bundle


In [4]:
sample_file = list(FHIR_DIR.glob("*.json"))[0]
bundle = json.load(open(sample_file))

print(f"File: {sample_file.name}")
print(f"Total entries in this bundle: {len(bundle['entry'])}")

# what resource types are inside?
types = Counter(e['resource']['resourceType'] for e in bundle['entry'])
print(f"\nResource types inside ONE patient file:")
for rtype, count in types.most_common():
    print(f"  {rtype}: {count}")

File: Adriana394_Guillermina633_Benítez559_91afa2e7-3e3f-1b50-6c96-400a2cd98d8f.json
Total entries in this bundle: 969

Resource types inside ONE patient file:
  Observation: 264
  Procedure: 192
  DiagnosticReport: 92
  Claim: 83
  ExplanationOfBenefit: 83
  Encounter: 59
  DocumentReference: 59
  Condition: 41
  MedicationRequest: 24
  SupplyDelivery: 16
  Immunization: 11
  Medication: 9
  MedicationAdministration: 9
  ImagingStudy: 8
  CareTeam: 6
  CarePlan: 6
  Device: 5
  Patient: 1
  Provenance: 1


### Functions to parse FHIR

In [5]:
def extract_observation_value(resource):
    """
    FHIR Observation uses polymorphic value[x]
    The field NAME changes based on data type
    """
    if "valueQuantity" in resource:
        # numeric — HbA1c, glucose, blood pressure
        return resource["valueQuantity"].get("value"), "numeric"
    
    if "valueString" in resource:
        # text — "Normal", "Abnormal"
        return resource["valueString"], "string"
    
    if "valueCodeableConcept" in resource:
        # coded — "Positive", "Negative"
        display = resource["valueCodeableConcept"]["coding"][0].get("display")
        return display, "codeable_concept"
    
    if "valueBoolean" in resource:
        # boolean — True/False
        return str(resource["valueBoolean"]), "boolean"
    
    return None, None


In [6]:
def parse_observation(resource):
    if resource.get("resourceType") != "Observation":
        return None
    
    # get LOINC code
    coding = resource.get("code", {}).get("coding", [])
    obs_code = coding[0].get("code") if coding else None
    obs_display = coding[0].get("display") if coding else None
    
    # get patient reference
    patient_ref = resource.get("subject", {}).get("reference", "")
    patient_id = patient_ref.split("/")[-1]
    
    # get value using our polymorphic handler
    value, value_type = extract_observation_value(resource)
    
    return {
        "patient_id": patient_id,
        "observation_code": obs_code,
        "observation_display": obs_display,
        "value": value,
        "value_type": value_type,
        "effective_date": resource.get("effectiveDateTime")
    }

In [7]:
def parse_condition(resource):
    if resource.get("resourceType") != "Condition":
        return None
    
    coding = resource.get("code", {}).get("coding", [])
    code = coding[0].get("code") if coding else None
    display = coding[0].get("display") if coding else None
    
    patient_ref = resource.get("subject", {}).get("reference", "")
    patient_id = patient_ref.replace("urn:uuid:", "").split("/")[-1]
    
    return {
        "patient_id": patient_id,
        "condition_code": code,
        "condition_display": display,
        "recorded_date": resource.get("recordedDate"),
        "clinical_status": resource.get("clinicalStatus", {})
                          .get("coding", [{}])[0].get("code")
    }

In [8]:
def parse_patient(resource):
    if resource.get("resourceType") != "Patient":
        return None
    
    birth_date = resource.get("birthDate")
    birth_year = int(birth_date[:4]) if birth_date else None
    
    return {
        "patient_id": resource.get("id"),
        "gender": resource.get("gender"),
        "birth_year": birth_year,
        "age": 2026 - birth_year if birth_year else None
    }

In [9]:
patients = []
conditions = []
observations = []

files = sorted(FHIR_DIR.glob("*.json"))[:15000]
print(f"Total files found: {len(files)}")

for i, file_path in enumerate(files):
    if i % 5000 == 0:
        print(f"  Processing {i}/{len(files)}...")
    
    try:
        with open(file_path) as f:
            bundle = json.load(f)
    except:
        continue
    
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})
        rtype = resource.get("resourceType")
        
        if rtype == "Patient":
            row = parse_patient(resource)
            if row:
                patients.append(row)
        
        elif rtype == "Observation":
            row = parse_observation(resource)
            if row:
                observations.append(row)


        elif rtype == "Condition":
            row = parse_condition(resource)
            if row:
                conditions.append(row)
patients_df = pd.DataFrame(patients).drop_duplicates(subset=["patient_id"])
observations_df = pd.DataFrame(observations)

print(f"\n Loaded:")
print(f"  Patients: {len(patients_df):,}")
print(f"  Observations: {len(observations_df):,}")

Total files found: 15000
  Processing 0/15000...
  Processing 5000/15000...
  Processing 10000/15000...

 Loaded:
  Patients: 14,945
  Observations: 6,908,746


### Transform to Parquet

In [10]:
observations_df["value_numeric"] = pd.to_numeric(
    observations_df["value"], errors="coerce"
)
observations_df["value_text"] = observations_df["value"].where(
    observations_df["value_type"].isin(["codeable_concept", "string"])
)
observations_df = observations_df.drop(columns=["value"])


patients_path = OUT_DIR / "patients.parquet"
observations_path = OUT_DIR / "observations.parquet"

observations_df["patient_id"] = observations_df["patient_id"].str.replace("urn:uuid:", "", regex=False)
observations_df.to_parquet(observations_path, index=False)

patients_df.to_parquet(patients_path, index=False)

In [11]:
print("=== Patients Table ===")
print(f"Shape: {patients_df.shape}")
print()

# clean display
sample_patients = patients_df[["patient_id", "gender", "birth_year", "age"]].head(5).copy()
sample_patients["patient_id"] =  "..."+ sample_patients["patient_id"].str[-8:] 

sample_patients.style\
    .set_properties(**{
        'background-color': '#1a1a1a',
        'color': 'white',
        'border-color': '#333',
        'font-size': '13px',
        'padding': '8px 12px'
    })\
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#2a2a2a'),
            ('color', '#00c896'),
            ('font-size', '13px'),
            ('padding', '8px 12px'),
            ('border-bottom', '2px solid #00c896')
        ]
    }])\
    .hide(axis='index')

=== Patients Table ===
Shape: (14945, 4)



patient_id,gender,birth_year,age
...33046e88,male,1960,66
...68ce778b,male,1985,41
...efcf03f1,male,1970,56
...1b9da1d0,male,1967,59
...97a46ffd,male,1966,60


In [12]:
# pick 3 different patients with different observation types
patient_ids = observations_df[
    observations_df["value_numeric"].notna()
]["patient_id"].unique()[:3]  # first 3 distinct patients

sample = observations_df[
    observations_df["patient_id"].isin(patient_ids) &
    observations_df["value_numeric"].notna()
][["patient_id", "observation_code", "observation_display", 
   "value_numeric", "value_type", "effective_date"]]\
.groupby("patient_id").head(2)\
.reset_index(drop=True)

sample = sample.copy()
sample["patient_id"] = "..." + sample["patient_id"].str[-8:]

sample.style\
    .set_properties(**{
        'background-color': '#1a1a1a',
        'color': 'white',
        'border-color': '#333',
        'font-size': '13px',
        'padding': '8px 12px'
    })\
    .set_table_styles([{
        'selector': 'th',
        'props': [
            ('background-color', '#2a2a2a'),
            ('color', '#00c896'),
            ('font-size', '13px'),
            ('padding', '8px 12px'),
            ('border-bottom', '2px solid #00c896')
        ]
    }])\
    .hide(axis='index')

patient_id,observation_code,observation_display,value_numeric,value_type,effective_date
...33046e88,8302-2,Body Height,175.600000,numeric,2016-11-30T22:11:38-08:00
...33046e88,72514-3,Pain severity - 0-10 verbal numeric rating [Score] - Reported,3.000000,numeric,2016-11-30T22:11:38-08:00
...68ce778b,8302-2,Body Height,182.800000,numeric,2016-10-17T07:58:00-07:00
...68ce778b,72514-3,Pain severity - 0-10 verbal numeric rating [Score] - Reported,2.000000,numeric,2016-10-17T07:58:00-07:00
...efcf03f1,2339-0,Glucose [Mass/volume] in Blood,72.250000,numeric,2017-04-26T10:34:51-07:00
...efcf03f1,6299-2,Urea nitrogen [Mass/volume] in Blood,10.070000,numeric,2017-04-26T10:34:51-07:00


In [13]:
conditions_df = pd.DataFrame(conditions)
conditions_df.to_parquet(OUT_DIR / "conditions.parquet", index=False)
print(f"Conditions: {len(conditions_df):,}")

Conditions: 510,235


In [15]:
#Input Paths

result = duckdb.sql("""
SELECT condition_display, COUNT(*) as patient_count
FROM read_parquet('puppygraph_output/conditions.parquet')
WHERE lower(condition_display) LIKE '%diabetes%'
   OR lower(condition_display) LIKE '%depression%'
   OR lower(condition_display) LIKE '%depress%'
GROUP BY condition_display
ORDER BY patient_count DESC
""").df()

print(result)
OUT_DIR.mkdir(exist_ok=True)

                                    condition_display  patient_count
0                               Prediabetes (finding)           5498
1   Disorder of kidney due to diabetes mellitus (d...           1268
2   Microalbuminuria due to type 2 diabetes mellit...           1057
3                 Diabetes mellitus type 2 (disorder)           1055
4   Proteinuria due to type 2 diabetes mellitus (d...            745
5   Neuropathy due to type 2 diabetes mellitus (di...            300
6   Nonproliferative diabetic retinopathy due to t...            239
7                Major depressive disorder (disorder)             80
8   Macular edema and retinopathy due to type 2 di...             30
9   Proliferative diabetic retinopathy due to type...             14
10  Diabetes mellitus due to cystic fibrosis (diso...              8
11        Major depression, single episode (disorder)              5
12  Blindness due to type 2 diabetes mellitus (dis...              2


In [18]:
result = duckdb.sql("""
WITH diabetes_patients AS (
    SELECT DISTINCT patient_id
    FROM read_parquet('puppygraph_output/conditions.parquet')
    WHERE lower(condition_display) LIKE '%diabetes mellitus type 2%'
),
depression_patients AS (
    SELECT DISTINCT patient_id
    FROM read_parquet('puppygraph_output/conditions.parquet')
    WHERE lower(condition_display) LIKE '%depression%'
)
SELECT COUNT(*) as both_conditions
FROM diabetes_patients d
JOIN depression_patients dep ON d.patient_id = dep.patient_id
""").fetchone()[0]

print(f"Patients with BOTH diabetes AND depression: {result}")

Patients with BOTH diabetes AND depression: 0


In [22]:
KEEP_CONDITIONS = [
    'diabetes', 'hypertension', 'obesity', 'depression',
    'anemia', 'cancer', 'failure', 'kidney', 'cardiac',
    'coronary', 'stroke', 'asthma', 'copd', 'anxiety',
    'osteoporosis', 'arthritis', 'prediabetes', 'neuropathy',
    'retinopathy', 'microalbuminuria', 'proteinuria'
]

conditions_filter = " OR ".join([
    f"lower(condition_display) LIKE '%{c}%'" 
    for c in KEEP_CONDITIONS
])

result = duckdb.sql(f"""
SELECT 
    c1.condition_display as condition_1,
    c2.condition_display as condition_2,
    COUNT(DISTINCT c1.patient_id) as patient_count
FROM read_parquet('puppygraph_output/conditions.parquet') c1
JOIN read_parquet('puppygraph_output/conditions.parquet') c2
    ON c1.patient_id = c2.patient_id
    AND c1.condition_display < c2.condition_display
WHERE 
    ({conditions_filter.replace('condition_display', 'c1.condition_display')})
    AND
    ({conditions_filter.replace('condition_display', 'c2.condition_display')})
GROUP BY c1.condition_display, c2.condition_display
ORDER BY patient_count DESC
LIMIT 15
""").df()

print(result)

                                          condition_1  \
0                                   Anemia (disorder)   
1             Body mass index 30+ - obesity (finding)   
2                                   Anemia (disorder)   
3             Body mass index 30+ - obesity (finding)   
4                   Essential hypertension (disorder)   
5                                   Anemia (disorder)   
6   Abnormal findings diagnostic imaging heart+cor...   
7   Abnormal findings diagnostic imaging heart+cor...   
8   Abnormal findings diagnostic imaging heart+cor...   
9   Abnormal findings diagnostic imaging heart+cor...   
10          Chronic kidney disease stage 1 (disorder)   
11  Disorder of kidney due to diabetes mellitus (d...   
12  Disorder of kidney due to diabetes mellitus (d...   
13            Body mass index 30+ - obesity (finding)   
14            Body mass index 30+ - obesity (finding)   

                                          condition_2  patient_count  
0               

In [23]:
import duckdb

# create a persistent duckdb file with views of your parquet files
con = duckdb.connect("puppygraph_output/healthcare.duckdb")

con.execute("""
    CREATE OR REPLACE VIEW patients AS 
    SELECT * FROM read_parquet('puppygraph_output/patients.parquet')
""")

con.execute("""
    CREATE OR REPLACE VIEW conditions AS 
    SELECT * FROM read_parquet('puppygraph_output/conditions.parquet')
""")

con.execute("""
    CREATE OR REPLACE VIEW observations AS 
    SELECT * FROM read_parquet('puppygraph_output/observations.parquet')
""")

con.close()
print(" DuckDB file created")

 DuckDB file created


In [ ]:
import duckdb

con = duckdb.connect("puppygraph_output/healthcare.duckdb")

con.execute("""
    CREATE OR REPLACE VIEW patients AS 
    SELECT * FROM read_parquet('puppygraph_output/patients.parquet')
""")

con.execute("""
    CREATE OR REPLACE VIEW conditions AS 
    SELECT * FROM read_parquet('puppygraph_output/conditions.parquet')
""")

con.execute("""
    CREATE OR REPLACE VIEW observations AS 
    SELECT * FROM read_parquet('puppygraph_output/observations.parquet')
""")

con.close()
print("Views updated with /data paths")

✓ Views updated with /data paths
